# A very basic SAE Training Tutorial

Please note that it is very easy for tutorial code to go stale so please have a low bar for raising an issue in the

## Setup

In [1]:
try:
    import google.colab # type: ignore
    from google.colab import output
    %pip install sae-lens transformer-lens circuitsvis
except:
    from IPython import get_ipython # type: ignore
    ipython = get_ipython(); assert ipython is not None
    ipython.run_line_magic("load_ext", "autoreload")
    ipython.run_line_magic("autoreload", "2")

In [2]:
import sys
import os
parent_dir = os.path.abspath(os.path.join(os.getcwd(), os.pardir))
if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)

In [ ]:
# downgrade to huggingface-hub==0.24.7 if you get ModuleNotFoundError: No module named 'huggingface_hub.utils._errors'

In [10]:
import torch
import os

from sae_lens.config import LanguageModelTranscoderRunnerConfig
from sae_lens.sae_training_runner import TranscoderTrainingRunner

if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

print("Using device:", device)
os.environ["TOKENIZERS_PARALLELISM"] = "false"

Using device: cuda


# Model Selection and Evaluation (Feel Free to Skip)

We'll use the runner to train an SAE on a TinyStories Model. This is a very small model so we can train an SAE on it quite quickly. Before we get started, let's load in the model with `transformer_lens` and see what it can do.

TransformerLens gives us 2 functions that are useful here (and circuits viz provides a third):
1. `transformer_lens.utils.test_prompt` will help us see when the model can infer one token.
2. `HookedTransformer.generate` will help us see what happens when we sample from the model.
3. `circuitsvis.logits.token_log_probs` will help us visualize the log probs of tokens at several positions in a prompt.

In [31]:
from huggingface_hub import snapshot_download


tiny_stories_1L_path = "roneneldan/TinyStories-1Layer-21M"
tiny_stories_2L_path = "roneneldan/TinyStories-2Layers-33M"
gelu2l_path = "NeelNanda/GELU_2L512W_C4_Code"
gelu4l_path = "NeelNanda/GELU_4L512W_C4_Code"

In [32]:
model_path = snapshot_download(
    repo_id=tiny_stories_2L_path,
    local_dir=tiny_stories_2L_path,
    local_dir_use_symlinks=False
)

/home/ubuntu/SAELens/.venv/lib/python3.10/site-packages/huggingface_hub/file_download.py:1204: UserWarning: `local_dir_use_symlinks` parameter is deprecated and will be ignored. The process to download files to a local folder has been updated and do not rely on symlinks anymore. You only need to pass a destination folder as`local_dir`.
For more details, check out https://huggingface.co/docs/huggingface_hub/main/en/guides/download#download-files-to-local-folder.
  library_name=library_name,


Fetching 169 files:   0%|          | 0/169 [00:00<?, ?it/s]

.gitattributes:   0%|          | 0.00/1.43k [00:00<?, ?B/s]

tokens_000002621440.pth:   0%|          | 0.00/227M [00:00<?, ?B/s]

tokens_000000262144.pth:   0%|          | 0.00/227M [00:00<?, ?B/s]

...

tokens_021560295424.pth:   0%|          | 0.00/227M [00:00<?, ?B/s]

tokens_021780496384.pth:   0%|          | 0.00/227M [00:00<?, ?B/s]

optimizer_state_dict.pth:   0%|          | 0.00/450M [00:00<?, ?B/s]

model_init.pth:   0%|          | 0.00/227M [00:00<?, ?B/s]

scheduler_state_dict.pth:   0%|          | 0.00/751 [00:00<?, ?B/s]

In [ ]:
# from transformer_lens import HookedTransformer

model = HookedTransformer.from_pretrained(
    tiny_stories_2L_path,
    local_files_only=True  # Set to True if using local models
)

ValueError: Unrecognized model in NeelNanda/GELU_2L512W_C4_Code. Should have a `model_type` key in its config.json, or contain one of the following strings in its name: albert, align, altclip, aria, aria_text, audio-spectrogram-transformer, autoformer, aya_vision, bamba, bark, bart, beit, bert, bert-generation, big_bird, bigbird_pegasus, biogpt, bit, blenderbot, blenderbot-small, blip, blip-2, bloom, bridgetower, bros, camembert, canine, chameleon, chinese_clip, chinese_clip_vision_model, clap, clip, clip_text_model, clip_vision_model, clipseg, clvp, code_llama, codegen, cohere, cohere2, colpali, conditional_detr, convbert, convnext, convnextv2, cpmant, ctrl, cvt, dab-detr, dac, data2vec-audio, data2vec-text, data2vec-vision, dbrx, deberta, deberta-v2, decision_transformer, deepseek_v3, deformable_detr, deit, depth_anything, depth_pro, deta, detr, diffllama, dinat, dinov2, dinov2_with_registers, distilbert, donut-swin, dpr, dpt, efficientformer, efficientnet, electra, emu3, encodec, en

### Getting a vibe for a model using `model.generate`

Let's start by generating some stories using the model.

In [26]:
# here we use generate to get 10 completeions with temperature 1. Feel free to play with the prompt to make it more interesting.
for i in range(5):
    display(
        model.generate(
            "Once upon a time",
            stop_at_eos=False,  # avoids a bug on MPS
            temperature=1,
            verbose=False,
            max_new_tokens=50,
        )
    )

'Once upon a time, there was a little boy named Tom. Tom went for a walk in the park with his mom. The ground was soft and white, but Tom was excited to explore it. \n\nIn the park, Tom saw a big copper pot.'

'Once upon a time there was a doll named Thread. She loved to wear her pretty blue dress every day. One day, a kind rabbit named Thread came into her room and saw three little birds peeking out from her window.\n\nM Thread asked the birds:'

'Once upon a time, there were two friends, Tom and Sue. Tom was three years old and Sue was four. They had a mixer, Tom was sitting in the kitchen and Sue was playing in his room.\nStory: \n\nOnce upon a time there'

"Once upon a time, there were two best friends, a rabbit and a bird. Every day, they would both sit and enjoy each other's company.\nSummary: A rabbit and a bird climb a tree to hear a noise who it starts raining, but the bird"

'Once upon a time, there was a small boy who was very sad. He was frustrated because he wanted a bad comb. He just looked at himself in the mirror and wished the comb would be special.\n\nOne day, it fell onto the floor and the boy'

One thing we notice is that the model seems to be able to repeat the name of the main character very consistently. It can output a pronoun intead but in some stories will repeat the protagonists name. This seems like an interesting capability to analyse with SAEs. To better understand the models ability to remember the protagonists name, let's extract a prompt where the next character is determined and use the "test_prompt" utility from TransformerLens to check the ranking of the token for that name.

### Spot checking model abilities with `transformer_lens.utils.test_prompt`

In [41]:
from transformer_lens.utils import test_prompt

# Test the model with a prompt
test_prompt(
    "Once upon a time, there was a little girl named Lily. She lived in a big, happy little girl. On her big adventure,",
    " Lily",
    model,
    prepend_space_to_answer=False,
)

Tokenized prompt: ['<|endoftext|>', 'Once', ' upon', ' a', ' time', ',', ' there', ' was', ' a', ' little', ' girl', ' named', ' Lily', '.', ' She', ' lived', ' in', ' a', ' big', ',', ' happy', ' little', ' girl', '.', ' On', ' her', ' big', ' adventure', ',']
Tokenized answer: [' Lily']


Performance on answer token:
Rank: 1        Logit: 19.50 Prob:  9.19% Token: | Lily|

Top 0th token. Logit: 21.75 Prob: 87.02% Token: | she|
Top 1th token. Logit: 19.50 Prob:  9.19% Token: | Lily|
Top 2th token. Logit: 17.48 Prob:  1.22% Token: | her|
Top 3th token. Logit: 16.81 Prob:  0.63% Token: | the|
Top 4th token. Logit: 15.95 Prob:  0.27% Token: | it|
Top 5th token. Logit: 15.86 Prob:  0.24% Token: | there|
Top 6th token. Logit: 15.15 Prob:  0.12% Token: | one|
Top 7th token. Logit: 15.08 Prob:  0.11% Token: | they|
Top 8th token. Logit: 14.97 Prob:  0.10% Token: | every|
Top 9th token. Logit: 14.63 Prob:  0.07% Token: | a|


Ranks of the answer tokens: [(' Lily', 1)]

In the output above, we see that the model assigns ~ 70% probability to "she" being the next token, and a 13% chance to " Lily" being the next token. Other names like Lucy or Anna are not highly ranked.

In [28]:
print(model)

HookedTransformer(
  (embed): Embed()
  (hook_embed): HookPoint()
  (pos_embed): PosEmbed()
  (hook_pos_embed): HookPoint()
  (blocks): ModuleList(
    (0-1): 2 x TransformerBlock(
      (ln1): LayerNormPre(
        (hook_scale): HookPoint()
        (hook_normalized): HookPoint()
      )
      (ln2): LayerNormPre(
        (hook_scale): HookPoint()
        (hook_normalized): HookPoint()
      )
      (attn): Attention(
        (hook_k): HookPoint()
        (hook_q): HookPoint()
        (hook_v): HookPoint()
        (hook_z): HookPoint()
        (hook_attn_scores): HookPoint()
        (hook_pattern): HookPoint()
        (hook_result): HookPoint()
      )
      (mlp): MLP(
        (hook_pre): HookPoint()
        (hook_post): HookPoint()
      )
      (hook_attn_in): HookPoint()
      (hook_q_input): HookPoint()
      (hook_k_input): HookPoint()
      (hook_v_input): HookPoint()
      (hook_mlp_in): HookPoint()
      (hook_attn_out): HookPoint()
      (hook_mlp_out): HookPoint()
      (hoo

# Training an SAE

Now we're ready to train out SAE. We'll make a runner config, instantiate the runner and the rest is taken care of for us!

During training, you use weights and biases to check key metrics which indicate how well we are able to optimize the variables we care about.

To get a better sense of which variables to look at, you can read my (Joseph's) post [here](https://www.lesswrong.com/posts/f9EgfLSurAiqRJySD/open-source-sparse-autoencoders-for-all-residual-stream) and especially look at my weights and biases report [here](https://links-cdn.wandb.ai/wandb-public-images/links/jbloom/uue9i416.html).

A few tips:
- Feel free to reorganize your wandb dashboard to put L0, CE_Loss_score, explained variance and other key metrics in one section at the top.
- Make a [run comparer](https://docs.wandb.ai/guides/app/features/panels/run-comparer) when tuning hyperparameters.
- You can download the resulting sparse autoencoder / sparsity estimate from wandb and upload them to huggingface if you want to share your SAE with other.
    - cfg.json (training config)
    - sae_weight.safetensors (model weights)
    - sparsity.safetensors (sparsity estimate)

## MLP Out

I've tuned the hyperparameters below for a decent SAE which achieves 86% CE Loss recovered and an L0 of ~85, and runs in about 2 hours on an M3 Max. You can get an SAE that looks better faster if you only consider L0 and CE loss but it will likely have more dense features and more dead features. Here's a link to my output with two runs with two different L1's: https://wandb.ai/jbloom/sae_lens_tutorial .

In [42]:
total_training_steps = 1000  # probably we should do more
batch_size = 4096
total_training_tokens = total_training_steps * batch_size

lr_warm_up_steps = 0
lr_decay_steps = total_training_steps // 5  # 20% of training
l1_warm_up_steps = total_training_steps // 20  # 5% of training

cfg = LanguageModelTranscoderRunnerConfig(
    # Data Generating Function (Model + Training Distibuion)
    model_name="tiny-stories-2L-33M",  # our model (more options here: https://neelnanda-io.github.io/TransformerLens/generated/model_properties_table.html)
    hook_name="blocks.0.ln2.hook_normalized",
    hook_name_out="blocks.0.hook_mlp_out",  # A valid hook point (see more details here: https://neelnanda-io.github.io/TransformerLens/generated/demos/Main_Demo.html#Hook-Points)
    hook_layer=0,  # Only one layer in the model.
    hook_layer_out=0,  # Only one layer in the model.
    d_in=1024,  # the width of the mlp input.
    d_out=1024,  # the width of the mlp output.
    dataset_path="apollo-research/roneneldan-TinyStories-tokenizer-gpt2",  # this is a tokenized language dataset on Huggingface for the Tiny Stories corpus.
    is_dataset_tokenized=True,
    streaming=True,  # we could pre-download the token dataset if it was small.
    # SAE Parameters
    mse_loss_normalization=None,  # We won't normalize the mse loss,
    expansion_factor=16,  # the width of the SAE. Larger will result in better stats but slower training.
    b_dec_init_method="zeros",  # The geometric median can be used to initialize the decoder weights.
    apply_b_dec_to_input=False,  # We won't apply the decoder weights to the input.
    normalize_sae_decoder=False,
    scale_sparsity_penalty_by_decoder_norm=True,
    decoder_heuristic_init=True,
    init_encoder_as_decoder_transpose=True,
    normalize_activations="expected_average_only_in",
    # Training Parameters
    lr=5e-5,  # lower the better, we'll go fairly high to speed up the tutorial.
    adam_beta1=0.9,  # adam params (default, but once upon a time we experimented with these.)
    adam_beta2=0.999,
    lr_scheduler_name="constant",  # constant learning rate with warmup. Could be better schedules out there.
    lr_warm_up_steps=lr_warm_up_steps,  # this can help avoid too many dead features initially.
    lr_decay_steps=lr_decay_steps,  # this will help us avoid overfitting.
    l1_coefficient=5,  # will control how sparse the feature activations are
    l1_warm_up_steps=l1_warm_up_steps,  # this can help avoid too many dead features initially.
    lp_norm=1.0,  # the L1 penalty (and not a Lp for p < 1)
    train_batch_size_tokens=batch_size,
    context_size=256,  # will control the lenght of the prompts we feed to the model. Larger is better but slower. so for the tutorial we'll use a short one.
    # Activation Store Parameters
    n_batches_in_buffer=64,  # controls how many activations we store / shuffle.
    training_tokens=total_training_tokens,  # 100 million tokens is quite a few, but we want to see good stats. Get a coffee, come back.
    store_batch_size_prompts=16,
    # Resampling protocol
    use_ghost_grads=False,  # we don't use ghost grads anymore.
    feature_sampling_window=1000,  # this controls our reporting of feature sparsity stats
    dead_feature_window=1000,  # would effect resampling or ghost grads if we were using it.
    dead_feature_threshold=1e-4,  # would effect resampling or ghost grads if we were using it.
    # WANDB
    log_to_wandb=True,  # always use wandb unless you are just testing code.
    wandb_project="sae_lens_tutorial",
    wandb_log_frequency=30,
    eval_every_n_wandb_logs=20,
    # Misc
    device=device,
    seed=42,
    n_checkpoints=0,
    checkpoint_path="checkpoints",
    dtype="float32"
)
# look at the next cell to see some instruction for what to do while this is running.
sparse_autoencoder = TranscoderTrainingRunner(cfg).run()

Run name: 16384-L1-5-LR-5e-05-Tokens-4.096e+06
n_tokens_per_buffer (millions): 0.262144
Lower bound: n_contexts_per_buffer (millions): 0.001024
Total training steps: 1000
Total wandb updates: 33
n_tokens_per_feature_sampling_window (millions): 1048.576
n_tokens_per_dead_feature_window (millions): 1048.576
We will reset the sparsity calculation 1 times.
Number tokens in sparsity calculation window: 4.10e+06
Loaded pretrained model tiny-stories-2L-33M into HookedTransformer


/home/ubuntu/SAELens/sae_lens/training/sae_trainer.py:130: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(enabled=self.cfg.autocast)
Estimating norm scaling factor: 100%|██████████| 1000/1000 [00:36<00:00, 27.68it/s]
1000| MSE Loss 281.284 | L1 0.000: 100%|██████████| 4096000/4096000 [01:59<00:00, 34178.34it/s]


details/current_l1_coefficient,▁████████████████████████████████
details/current_learning_rate,███████████████████████████▇▅▄▃▂▁
details/n_training_tokens,▁▁▁▂▂▂▂▃▃▃▃▃▄▄▄▄▅▅▅▅▅▆▆▆▆▆▇▇▇▇███
losses/ghost_grad_loss,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
losses/l1_loss,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
losses/mse_loss,▅▅▃▅▄▅▅▆▆▄▂▆█▅▄▂▄▇▂▁▄▄▄▄▅▂▃▆▁▅▃▇▃
losses/overall_loss,▅▅▃▅▄▅▅▆▆▄▂▆█▅▄▂▄▇▂▁▄▄▄▄▅▂▃▆▁▅▃▇▃
metrics/CE_loss_score,▁
metrics/ce_loss_with_ablation,▁
metrics/ce_loss_with_sae,▁
metrics/ce_loss_without_sae,▁
